# Credit Card Fraud Detection — CRISP-DM

**Business objective:** Detect suspicious transactions while balancing fraud losses against investigation workload. Dataset: [ULB/Worldline credit card fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

Run all cells with `creditcard.csv` in `data/`. This original, documented Jupyter notebook follows all six phases of the **CRISP-DM** process.

**Published article:** [99.83% Accurate. Zero Fraud Caught. Here's the Catch.](https://medium.com/@ayobamiawogboro123/99-83-accurate-zero-fraud-caught-heres-the-catch-4723a7b49b1c).

## 1. Business Understanding — four research questions

1. **Q1: Fraud prevalence.** How rare is fraud, and why can headline accuracy mislead?
2. **Q2: Transaction behavior.** How do transaction amounts vary between fraud and legitimate transactions?
3. **Q3: Predictive performance.** How do Logistic Regression and Random Forest compare for recall, precision, and false alerts?
4. **Q4: Decision support.** What does a model predict for a transaction it has not seen during training?

This is a fraud-investigation proof of concept, not an approved production decision system.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
REQUIRED_COLUMNS = {"Time", "Amount", "Class"}
DATA_CANDIDATES = [
    Path("../data/creditcard.csv"),
    Path("data/creditcard.csv"),
]
IMAGE_DIRECTORY = (
    Path("../images") if Path("../data").is_dir() else Path("images")
)
IMAGE_DIRECTORY.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
def load_transactions(candidate_paths):
    """Read the source CSV and validate the required fraud target."""
    source_path = next((p for p in candidate_paths if p.is_file()), None)
    if source_path is None:
        raise FileNotFoundError(
            "Download creditcard.csv from the Kaggle source in README.md "
            "and place it in data/creditcard.csv."
        )
    transactions = pd.read_csv(source_path)
    if not REQUIRED_COLUMNS.issubset(transactions.columns):
        raise ValueError("Source dataset is missing required columns.")
    if not transactions["Class"].dropna().isin([0, 1]).all():
        raise ValueError("Class must contain binary values (0, 1).")
    return transactions


def clean_transactions(transactions):
    """Remove duplicate and incomplete records before model fitting."""
    return transactions.drop_duplicates().dropna().copy()


def save_figure(figure, filename):
    """Export one publication-quality Matplotlib chart to images/."""
    figure.tight_layout()
    figure.savefig(
        IMAGE_DIRECTORY / filename,
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()


raw_transactions = load_transactions(DATA_CANDIDATES)
print("Original transactions:", len(raw_transactions))
print("Missing values:", int(raw_transactions.isna().sum().sum()))
print("Exact duplicate records:", int(raw_transactions.duplicated().sum()))
display(raw_transactions.head())

## 2. Data Understanding / EDA

Anonymized PCA variables `V1`–`V28`, numeric `Time` and `Amount`, and binary `Class` are supplied. The original data does **not** identify currency, merchants, customer identities, or locations.

### Q1 — How prevalent is fraud?

Use class counts, prevalence, and a log-scale count plot. A majority-only classifier catches no fraud regardless of its accuracy.

In [ ]:
class_counts = raw_transactions["Class"].value_counts().sort_index()
fraud_rate = class_counts.loc[1] / class_counts.sum()
baseline_accuracy = class_counts.loc[0] / class_counts.sum()

class_summary = pd.DataFrame(
    {
        "Transaction type": ["Legitimate", "Fraud"],
        "Transaction count": class_counts.values,
        "Share (%)": 100 * class_counts.values / class_counts.sum(),
    }
)
display(class_summary)
print(f"Q1 answer: fraud incidence is {fraud_rate:.3%}.")
print(
    f"Majority-only accuracy: {baseline_accuracy:.2%}; "
    "majority-only fraud recall: 0%."
)

figure, axis = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=class_summary,
    x="Transaction type",
    y="Transaction count",
    color="#4677ad",
    ax=axis,
)
axis.set_yscale("log")
axis.set_title("Fraud vs. legitimate transactions (log scale)")
for index, count in enumerate(class_counts.values):
    axis.text(index, count * 1.25, f"{count:,}", ha="center")
save_figure(figure, "fraud_distribution.png")

### Q2 — How do transaction amounts differ?

Compare class-specific amount summaries and distributions. A `log1p` transform is used **only for visualization** to show right-skew; raw amounts are retained for training.

In [ ]:
amount_summary = raw_transactions.groupby("Class")["Amount"].agg(
    ["count", "median", "mean", "max"]
)
display(amount_summary.round(2))

plot_transactions = raw_transactions.assign(
    log_amount=np.log1p(raw_transactions["Amount"]),
    class_label=raw_transactions["Class"].map(
        {0: "Legitimate", 1: "Fraud"}
    ),
)
figure, axis = plt.subplots(figsize=(9, 4))
sns.histplot(
    data=plot_transactions,
    x="log_amount",
    hue="class_label",
    bins=55,
    stat="density",
    common_norm=False,
    element="step",
    ax=axis,
)
axis.set_xlabel("log(1 + transaction amount)")
axis.set_title("Transaction amount by label")
save_figure(figure, "transaction_amount_distribution.png")
print("Q2 answer: median amount (fraud, legitimate) is 9.25 vs 22.00.")
print("Amount is not a reliable stand-alone fraud rule.")

In [ ]:
figure, axis = plt.subplots(figsize=(11, 8))
sns.heatmap(
    raw_transactions.corr(numeric_only=True),
    center=0,
    cmap="coolwarm",
    ax=axis,
)
axis.set_title("Feature correlation heatmap")
save_figure(figure, "correlation_heatmap.png")

## 3. Data Preparation — justify cleaning and encoding

- **Missing data:** source contains zero missing values. No imputation needed; incomplete records would be removed as a defensive safeguard.
- **Duplicates:** 1,081 exact repeated records were found and removed to avoid double-counting observations.
- **Categorical predictors:** none; all input variables are numeric. The binary target is already encoded as 0/1, so no one-hot encoding is appropriate.
- **Imbalance:** stratified 80/20 split maintains fraud prevalence. Class weighting replaces synthetic oversampling.
- **Leakage:** StandardScaler is inside a Pipeline fitted only on training data.

In [ ]:
transactions = clean_transactions(raw_transactions)
removed_count = len(raw_transactions) - len(transactions)
features = transactions.drop(columns="Class")
target = transactions["Class"].astype(int)

assert all(
    pd.api.types.is_numeric_dtype(dtype)
    for dtype in features.dtypes
)
assert target.nunique() == 2

train_features, test_features, train_target, test_target = (
    train_test_split(
        features,
        target,
        test_size=0.20,
        stratify=target,
        random_state=RANDOM_STATE,
    )
)
print(f"Cleaned rows: {len(transactions):,}; removed: {removed_count:,}")
print(f"Train: {len(train_features):,}; test: {len(test_features):,}")
print(f"Fraud cases in test: {int(test_target.sum()):,}")

## 4. Modeling

### Q3 — Which model makes more useful fraud alerts?

Compare standardized, balanced Logistic Regression with a 60-tree balanced Random Forest. No tuning on the held-out test set; both retain default decision thresholds.

In [ ]:
def build_models(random_state):
    """Create reproducible, class-weighted binary fraud classifiers."""
    return {
        "Logistic Regression": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "classifier",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        random_state=random_state,
                    ),
                ),
            ]
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=60,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1,
        ),
    }


def evaluate_classifier(model, test_data, true_labels):
    """Calculate held-out classification metrics and prediction scores."""
    predictions = model.predict(test_data)
    scores = model.predict_proba(test_data)[:, 1]
    metrics = {
        "Accuracy": accuracy_score(true_labels, predictions),
        "Precision": precision_score(
            true_labels, predictions, zero_division=0
        ),
        "Recall": recall_score(true_labels, predictions, zero_division=0),
        "F1": f1_score(true_labels, predictions, zero_division=0),
        "Average precision": average_precision_score(true_labels, scores),
        "ROC AUC": roc_auc_score(true_labels, scores),
    }
    return metrics, predictions, scores


models = build_models(RANDOM_STATE)
model_rows = []
model_predictions = {}
model_scores = {}

for model_name, classifier in models.items():
    classifier.fit(train_features, train_target)
    metrics, predictions, scores = evaluate_classifier(
        classifier, test_features, test_target
    )
    model_rows.append({"Model": model_name, **metrics})
    model_predictions[model_name] = predictions
    model_scores[model_name] = scores

results_table = pd.DataFrame(model_rows).set_index("Model")
display(results_table.round(4))

## 5. Evaluation: explain the business consequences

**Recall:** how many actual fraud cases were caught. **Precision:** how many raised alerts were fraud. **F1:** precision/recall balance at the default threshold. **Average precision:** summarizes performance across thresholds. **Accuracy** is not sufficient with 0.173% fraud. Show error types in confusion matrices and precision-recall curves.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, (model_name, predictions) in zip(
    axes, model_predictions.items()
):
    ConfusionMatrixDisplay.from_predictions(
        test_target,
        predictions,
        ax=axis,
        colorbar=False,
        display_labels=["Legitimate", "Fraud"],
    )
    axis.set_title(model_name)
save_figure(figure, "confusion_matrices.png")

figure, axis = plt.subplots(figsize=(7, 5))
for model_name, scores in model_scores.items():
    PrecisionRecallDisplay.from_predictions(
        test_target, scores, name=model_name, ax=axis
    )
axis.set_title("Precision-recall curves")
save_figure(figure, "precision_recall_curves.png")

figure, axis = plt.subplots(figsize=(8, 4))
results_table[["Precision", "Recall", "F1", "Average precision"]].plot(
    kind="bar", ax=axis, rot=0
)
axis.set_ylim(0, 1)
axis.set_title("Fraud classification: held-out comparison")
save_figure(figure, "model_comparison.png")

In [ ]:
error_summary = {}
for model_name, predictions in model_predictions.items():
    true_negative, false_positive, false_negative, true_positive = (
        confusion_matrix(test_target, predictions, labels=[0, 1]).ravel()
    )
    error_summary[model_name] = {
        "Fraud detected": int(true_positive),
        "False alerts": int(false_positive),
        "Fraud missed": int(false_negative),
    }

display(pd.DataFrame(error_summary).T)
print("Q3 answer:")
print("Logistic Regression: 83/95 fraud caught, 1,389 false alerts.")
print("Random Forest: 69/95 fraud caught, 2 false alerts.")
print("No operational winner without investigation and loss costs.")

## 6. Deployment Demonstration

### Q4 — What does an unseen transaction prediction tell us?

Use the first held-out test transaction to obtain authentic anonymized features rather than inventing impossible PCA inputs. For illustration, choose the highest average-precision model **after** evaluation; this is post-hoc model selection, not independent validation. Model scores are **uncalibrated** and must not be interpreted as confirmed probabilities. The output supports analyst triage, not automatic payment rejection.

In [ ]:
selected_name = results_table["Average precision"].idxmax()
selected_model = models[selected_name]
new_transaction = test_features.iloc[[0]].copy()
actual_label = int(test_target.iloc[0])
model_score = float(
    selected_model.predict_proba(new_transaction)[0, 1]
)
predicted_label = int(selected_model.predict(new_transaction)[0])

print(f"Demonstration model: {selected_name}")
print(f"Transaction amount: {new_transaction['Amount'].iloc[0]:.2f}")
print(f"Uncalibrated score: {model_score:.4f}")
print(f"Predicted label: {predicted_label}; actual: {actual_label}")
print(
    "Q4 answer:",
    "Review this alert" if predicted_label else "Not flagged by model",
)
print("A single correct test prediction is not a deployment guarantee.")

## Four Business Questions — Answered

| Question | Evidence-backed answer |
|---|---|
| **Q1. Incidence** | 492 of 284,807 original transactions (0.173%) are fraud. A trivial majority predictor has 99.83% accuracy and **zero** fraud recall. |
| **Q2. Amount** | Original medians: 9.25 for fraud, 22.00 for legitimate transactions (currency unspecified). Amount alone is not a reliable causal fraud indicator. |
| **Q3. Models** | Logistic Regression detected 83 of 95 test fraud cases with 1,389 false alerts; Random Forest detected 69 with 2 false alerts. |
| **Q4. Prediction** | The selected Random Forest classified a genuine held-out transaction as legitimate. It is one screening example, not a bank-ready decision process. |

**Limitations and improvements:** Random stratification does not test temporal drift; source PCA features prevent business-level feature interpretation. Model scores require calibration. Production use would require chronological validation, separate threshold selection, cost analysis, privacy/governance review, drift monitoring, and human oversight.

**Acknowledgments:** ULB Machine Learning Group / Worldline (dataset via Kaggle); pandas, NumPy, scikit-learn, Matplotlib, Seaborn, and the Python open-source community.